In this notebook I will join on covariates to the initial basefile

In [1]:
import requests
import pandas as pd

def fetch_bikepoint_locations() -> pd.DataFrame:
    url = "https://api.tfl.gov.uk/BikePoint"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()

    rows = []
    for s in data:
        # BikePoints_123 -> 123
        sid = s["id"].split("_")[1]
        rows.append({
            "station_id": int(sid),
            "station_name": s.get("commonName"),
            "lat": float(s.get("lat")),
            "lon": float(s.get("lon")),
        })

    stations = pd.DataFrame(rows).dropna(subset=["station_id", "lat", "lon"]).drop_duplicates("station_id")
    return stations

stations = fetch_bikepoint_locations()
stations.head()

,station_id,station_name,lat,lon
0,1,"River Street , Clerkenwell",51.529163,-0.109970
1,2,"Phillimore Gardens, Kensington",51.499606,-0.197574
2,3,"Christopher Street, Liverpool Street",51.521283,-0.084605
3,4,"St. Chad's Street, King's Cross",51.530059,-0.120973
4,5,"Sedding Street, Sloane Square",51.493130,-0.156876


In [2]:
stations.to_csv("data\\bike_stations.csv", index=False)

In [1]:
import pyarrow.parquet as pa

_base = pa.read_table('E:\\tfl_project\outputs\\base_station_hour.parquet')
bf = _base.to_pandas()
bf.head()

,station_id,ts,trips_start,strike_exposed
0,1,2016-01-10 09:00:00,4,0
1,1,2016-01-10 10:00:00,1,0
2,1,2016-01-10 11:00:00,2,0
3,1,2016-01-10 12:00:00,2,0
4,1,2016-01-10 13:00:00,2,0


In [4]:
bf.shape

(9671772, 4)

In [5]:
bf["station_id"] = bf["station_id"].astype(int)
stations["station_id"] = stations["station_id"].astype(int)

base = bf.merge(stations[["station_id", "lat", "lon", "station_name"]],
                  on="station_id", how="left")

# sanity check
base[["station_id", "lat", "lon"]].isna().mean()

station_id    0.000000
lat           0.049022
lon           0.049022
dtype: float64

Here I removing any stations that don't appear in the BikePoint Metadata. This is only 5% of rows

“Approximately 5% of station-hour observations corresponded to stations without matching BikePoint metadata (likely decommissioned or renamed docks). These were excluded from the spatial analysis. Results were robust to their inclusion when geographic covariates were omitted.”

In [6]:
base_ids = set(base["station_id"].unique())
station_ids = set(stations["station_id"].unique())

missing_ids = base_ids - station_ids

#missing_ids

In [7]:
base_filtered = base[~base["station_id"].isin(missing_ids)].copy()

In [8]:
base_filtered.shape

(9197640, 7)

In [9]:
base_filtered.head()

,station_id,ts,trips_start,strike_exposed,lat,lon,station_name
0,1,2016-01-10 09:00:00,4,0,51.529163,-0.10997,"River Street , Clerkenwell"
1,1,2016-01-10 10:00:00,1,0,51.529163,-0.10997,"River Street , Clerkenwell"
2,1,2016-01-10 11:00:00,2,0,51.529163,-0.10997,"River Street , Clerkenwell"
3,1,2016-01-10 12:00:00,2,0,51.529163,-0.10997,"River Street , Clerkenwell"
4,1,2016-01-10 13:00:00,2,0,51.529163,-0.10997,"River Street , Clerkenwell"


In [11]:
#save this parquet
base_filtered.to_parquet( "base_station_hour_coords.parquet", index=False)


In [7]:
import polars as pl

df = pl.read_parquet("base_station_hour_coords.parquet")

# Swap names to match the pipeline's expectation:
#   trips_start -> timestamp
#   ts         -> trip count
df = df.rename({
    "ts": "trips_start",
    "trips_start": "ts",
})

df.write_parquet("base_station_hour_coords_renamed.parquet")

In [1]:
import polars as pl

df = pl.read_parquet("base_station_hour_coords_renamed.parquet")

df

station_id,trips_start,ts,strike_exposed,lat,lon,station_name
i64,datetime[μs],i32,i64,f64,f64,str
1,2016-01-10 09:00:00,4,0,51.529163,-0.10997,"""River Street , Clerkenwell"""
1,2016-01-10 10:00:00,1,0,51.529163,-0.10997,"""River Street , Clerkenwell"""
1,2016-01-10 11:00:00,2,0,51.529163,-0.10997,"""River Street , Clerkenwell"""
1,2016-01-10 12:00:00,2,0,51.529163,-0.10997,"""River Street , Clerkenwell"""
1,2016-01-10 13:00:00,2,0,51.529163,-0.10997,"""River Street , Clerkenwell"""
…,…,…,…,…,…,…
839,2018-12-24 01:00:00,1,0,51.508033,-0.106823,"""Sea Containers, South Bank"""
839,2018-12-24 12:00:00,1,0,51.508033,-0.106823,"""Sea Containers, South Bank"""
839,2018-12-24 14:00:00,1,0,51.508033,-0.106823,"""Sea Containers, South Bank"""


In [2]:
from covariate_join_utils_v2 import build_bike_panel_with_covariates_from_panel

# out = build_bike_panel_with_covariates_from_panel(
#     bike_path="base_station_hour_coords_renamed.parquet",
#     out_path="base_station_hour_coords_with_covariates.parquet",
#     grid_km=1.1,
#     include_daylight=True,  # pip install astral pytz
#     start_date="2016-01-01",
#     end_date="2018-12-31",
# )
#out.head()

In [3]:
from covariate_join_utils_v2 import fetch_osm_cycle_lanes
fetch_osm_cycle_lanes(out_path="osm_cycle_lanes.json")

Saved 38633 OSM ways → osm_cycle_lanes.json


In [4]:
build_bike_panel_with_covariates_from_panel(
      bike_path     = "base_station_hour_coords_renamed.parquet",
      out_path      = "bike_hourly_with_covariates_v2.parquet",
      start_date    = "2016-01-01",
      end_date      = "2018-12-31",
      grid_km       = 1.1,
      osm_json_path = "osm_cycle_lanes.json",   # omit to skip cycle_infra_score
      include_daylight = True,                   # pip install astral pytz
  )

In [5]:
import polars as pl

df = pl.read_parquet("bike_hourly_with_covariates_v2.parquet")

df.head()

station_id,trips_start,date,ts,y_log1p,strike_exposed,lat,lon,station_name,weather_cell_id,hour,dow,month,year,doy,is_weekend,is_am_peak,is_pm_peak,is_bank_holiday,is_school_holiday,temperature_2m,relative_humidity_2m,precipitation,rain,cloud_cover,wind_speed_10m,weather_code,dist_nearest_tube_km,n_tube_within_500m,n_tube_within_1km,strike_severity_daily_frac,days_to_next_strike,days_since_last_strike,cycle_infra_score
i32,datetime[μs],date,i32,f64,i8,f64,f64,str,str,i8,i8,i8,i32,i16,i8,i8,i8,i8,i8,f64,i64,f64,f64,i64,f64,i64,f64,i64,i64,f64,i64,i64,i32
39,2016-08-18 10:00:00,2016-08-18,3,1.386294,0,51.526377,-0.07813,"""Shoreditch High Street, Shored…","""g1.100_lat5214_lon-5""",10,4,8,2016,231,0,0,0,0,1,20.3,71,0.0,0.0,94,8.3,3,0.339435,1,3,0.0,30,30,188
39,2016-08-18 11:00:00,2016-08-18,4,1.609438,0,51.526377,-0.07813,"""Shoreditch High Street, Shored…","""g1.100_lat5214_lon-5""",11,4,8,2016,231,0,0,0,0,1,21.0,67,0.0,0.0,92,10.4,3,0.339435,1,3,0.0,30,30,188
39,2016-08-18 12:00:00,2016-08-18,7,2.079442,0,51.526377,-0.07813,"""Shoreditch High Street, Shored…","""g1.100_lat5214_lon-5""",12,4,8,2016,231,0,0,0,0,1,22.1,61,0.0,0.0,74,10.6,2,0.339435,1,3,0.0,30,30,188
39,2016-08-18 13:00:00,2016-08-18,6,1.94591,0,51.526377,-0.07813,"""Shoreditch High Street, Shored…","""g1.100_lat5214_lon-5""",13,4,8,2016,231,0,0,0,0,1,22.6,58,0.0,0.0,66,9.4,2,0.339435,1,3,0.0,30,30,188
39,2016-08-18 14:00:00,2016-08-18,1,0.693147,0,51.526377,-0.07813,"""Shoreditch High Street, Shored…","""g1.100_lat5214_lon-5""",14,4,8,2016,231,0,0,0,0,1,22.7,57,0.0,0.0,46,11.5,1,0.339435,1,3,0.0,30,30,188


In [6]:
df.shape

(9197640, 34)

We want to filter the basefile based on if the bike station is near to a tube station

We do this because we don't want to break the overlap assumption - basically every unit has to have a non zero probability of having the treatment applied

If a bike station is not close to a tube station we will remove it

In [10]:
import pyarrow.parquet as pa

_base = pa.read_table("bike_hourly_with_covariates_v2.parquet")
bf = _base.to_pandas()
bf.head()

,station_id,trips_start,date,ts,y_log1p,strike_exposed,lat,lon,station_name,weather_cell_id,...,cloud_cover,wind_speed_10m,weather_code,dist_nearest_tube_km,n_tube_within_500m,n_tube_within_1km,strike_severity_daily_frac,days_to_next_strike,days_since_last_strike,cycle_infra_score
0,39,2016-08-18 10:00:00,2016-08-18,3,1.386294,0,51.526377,-0.07813,"Shoreditch High Street, Shoreditch",g1.100_lat5214_lon-5,...,94.0,8.3,3.0,0.339435,1,3,0.0,30.0,30.0,188
1,39,2016-08-18 11:00:00,2016-08-18,4,1.609438,0,51.526377,-0.07813,"Shoreditch High Street, Shoreditch",g1.100_lat5214_lon-5,...,92.0,10.4,3.0,0.339435,1,3,0.0,30.0,30.0,188
2,39,2016-08-18 12:00:00,2016-08-18,7,2.079442,0,51.526377,-0.07813,"Shoreditch High Street, Shoreditch",g1.100_lat5214_lon-5,...,74.0,10.6,2.0,0.339435,1,3,0.0,30.0,30.0,188
3,39,2016-08-18 13:00:00,2016-08-18,6,1.945910,0,51.526377,-0.07813,"Shoreditch High Street, Shoreditch",g1.100_lat5214_lon-5,...,66.0,9.4,2.0,0.339435,1,3,0.0,30.0,30.0,188
4,39,2016-08-18 14:00:00,2016-08-18,1,0.693147,0,51.526377,-0.07813,"Shoreditch High Street, Shoreditch",g1.100_lat5214_lon-5,...,46.0,11.5,1.0,0.339435,1,3,0.0,30.0,30.0,188


In [11]:
bf[['dist_nearest_tube_km']].describe()

,dist_nearest_tube_km
count,9.197640e+06
mean,9.937210e-01
std,9.790543e-01
min,2.046157e-02
25%,3.216469e-01
50%,5.928447e-01
75%,1.343625e+00
max,4.747068e+00


In [13]:
bf_filtered = bf[bf['dist_nearest_tube_km'] < 2]

In [15]:
bf_filtered.to_parquet("bike_hourly_with_covariates_v2_filtered.parquet")